# 11 Customer deployment
Stand the assistant up for a second, simulated customer: their pages, their roles, a leak test per role, an audit log, a field log, and one change from a recurring field log item.
Everything is written to `stage-11/`. Save your run of this notebook as `stage-11/notebooks/11_customer_deployment.ipynb`.

Step 1: set up the kit

In [1]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-11"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Kit folder found: atp-builder-kit


Step 2: read the customer brief, and name the model provider

In [2]:
import json, re, time
from datetime import date, datetime, timezone
import pandas as pd
from kit.llm import call_llm, default_model, pick_provider

# Northwind Learning is invented practice content: an online tutoring firm with three roles.
CUSTOMER = Path(os.environ.get("CUSTOMER_DIR", ROOT / "data" / "customer_northwind"))
brief = (CUSTOMER / "BRIEF.md").read_text()
print("\n".join(brief.splitlines()[:24]))
PROVIDER = pick_provider()
MODEL = default_model(PROVIDER)
print(f"\nModel provider for this run: {PROVIDER}, model: {MODEL}")
if PROVIDER == "mock":
    print("Offline mock model: the numbers below only show the notebook works. Run it live for your README.")

# Customer brief: Northwind Learning (simulated)

Northwind Learning is an invented company for practice. It is a small online tutoring firm, not a real
business. Everything in this folder is made up for the course.

## Who they are

- About 40 tutors and 900 students, mostly high school maths and science.
- Lessons run as video calls booked through the Northwind web app.
- Support today is two people answering email. Most questions repeat every week.

## What they want

A help assistant inside their app that answers from their own help pages, so students and tutors get
an answer in seconds and the support team keeps only the hard cases.

## Who uses it (three roles)

| Role | Who | May read |
| --- | --- | --- |
| student | a learner or their parent | student pages |
| tutor | a tutor on contract | student and tutor pages |
| admin | the support and operations team | every page |


Model provider for this run: mock, model: mock-model
Offline mock model: the numbers below only show the

Step 3: re-point retrieval at their pages, with their roles

In [3]:
from kit.rag import VectorIndex, chunk_pages, get_embedder, parse_corpus

# Their roles, not Quillnote's: for each page visibility, the roles that may read it.
NW_ACCESS = {"student": ["student", "tutor", "admin"], "tutor": ["tutor", "admin"], "admin": ["admin"]}
ROLES = ["student", "tutor", "admin"]
pages = parse_corpus(CUSTOMER / "pages")
chunks = chunk_pages(pages, 120, 30, access=NW_ACCESS)
index = VectorIndex(chunks, get_embedder())
print(len(pages), "pages,", len(chunks), "chunks, embedder:", index.embedder.name)
pd.DataFrame(pages)[["id", "title", "visibility"]]

10 pages, 10 chunks, embedder: hash-384


,id,title,visibility
0,nw-01,Book a tutoring session,student
1,nw-02,Reschedule or cancel a session,student
2,nw-03,Join a video lesson,student
3,nw-04,Upload homework and read feedback,student
4,nw-05,Set your tutor availability,tutor
5,nw-06,Tutor pay and timesheets,tutor
6,nw-07,Write session notes and progress reports,tutor
7,nw-08,Refund a missed or cancelled session,admin
8,nw-09,Add or remove a tutor,admin
9,nw-10,Safeguarding reports,admin


Step 4: answer their questions as each role

Every answer also writes one line to `stage-11/audit-log.jsonl`: time, role, question id, pages retrieved and cited,
provider, model and tokens. The audit log keeps no raw question or answer text (the redaction rule for the procurement pack).
Raw answers for this practice run go to `stage-11/answers.jsonl`: the questions are synthetic, so they are safe to commit.

In [4]:
SYSTEM = ("Answer using only the numbered sources. Cite the source ids you used in square brackets, like [nw-02#0]. "
          "If the sources do not answer the question, say so and tell the user to contact support.")
DISCLOSURE = "You are chatting with an AI assistant. It answers only from Northwind help pages and can be wrong."
questions = [json.loads(line) for line in open(CUSTOMER / "questions.jsonl")]
AUDIT = OUTPUT / "audit-log.jsonl"
AUDIT.unlink(missing_ok=True)

def audit(action, role, qid, retrieved, cited, call):
    rec = {"time": datetime.now(timezone.utc).isoformat(timespec="seconds"), "action": action, "role": role,
           "qid": qid, "retrieved": retrieved, "cited": sorted(cited), "provider": call["provider"],
           "model": call["model"], "input_tokens": call["input_tokens"], "output_tokens": call["output_tokens"]}
    with open(AUDIT, "a") as f:
        f.write(json.dumps(rec) + "\n")

def answer(question, role, qid="adhoc"):
    hits = index.search(question, k=3, role=role)  # the role filter from Chapter 9, with their roles
    sources = "\n".join(f"[{c['id']}] {c['text']}" for c, _ in hits)
    r = call_llm(f"Sources:\n{sources}\n\nQuestion: {question}", system=SYSTEM, max_output_tokens=1000)
    cited = {c.split("#")[0] for c in re.findall(r"nw-\d+(?:#\d+)?", r["text"])}
    retrieved = [c["page_id"] for c, _ in hits]
    audit("answer", role, qid, retrieved, cited, r)
    return r["text"], cited, retrieved

print("Shown to every user above the chat box:", DISCLOSURE)
rows = []
for q in questions:
    text, cited, retrieved = answer(q["question"], q["role"], q["qid"])
    rows.append({"qid": q["qid"], "role": q["role"], "question": q["question"], "expected": q["page_ids"],
                 "retrieved": retrieved, "cited": sorted(cited), "correct": bool(cited & set(q["page_ids"])),
                 "answer": text})
results = pd.DataFrame(rows)
results.to_json(OUTPUT / "answers.jsonl", orient="records", lines=True)
print(results.groupby("role", sort=False).correct.agg(["sum", "count"]))
print(f"Answered from the right page: {results.correct.sum()} of {len(results)}. Raw answers in answers.jsonl,"
      f" {sum(1 for _ in open(AUDIT))} audit lines in audit-log.jsonl")
results[~results.correct][["qid", "role", "question", "expected", "retrieved"]]

Shown to every user above the chat box: You are chatting with an AI assistant. It answers only from Northwind help pages and can be wrong.
         sum  count
role               
student    5      7
tutor      3      3
admin      2      2
Answered from the right page: 10 of 12. Raw answers in answers.jsonl, 12 audit lines in audit-log.jsonl


,qid,role,question,expected,retrieved
5,n06,student,My tutor cancelled. Do I get a refund?,[nw-02],"[nw-04, nw-03, nw-01]"
6,n07,student,Can my parent see the session notes from my tu...,[nw-07],"[nw-02, nw-04, nw-03]"


Step 5: the leak test, one run per customer role

For each role, every prompt is searched with that role's filter. A prompt passes when every page retrieved is one the
role may read. Raw results per role go to `stage-11/leak-test/raw-<role>.jsonl` and the summary table to
`stage-11/leak-test/summary.md`. A control run with the filter off must fail, or the test proves nothing.

In [5]:
LEAK = OUTPUT / "leak-test"
LEAK.mkdir(exist_ok=True)
VISIBILITY = {p["id"]: p["visibility"] for p in pages}
probes = [q["question"] for q in questions]
probes += ["tutor pay rate", "refund a late cancel", "safeguarding report details", "deactivate a tutor"]

def may_read(role, visibility):
    return role in NW_ACCESS[visibility]

def leak_run(role, label, k=5, filter_role=True):
    raw = []
    for p in probes:
        hits = index.search(p, k=k, role=role if filter_role else None)
        seen = [{"page_id": c["page_id"], "visibility": VISIBILITY[c["page_id"]]} for c, _ in hits]
        leaked = [s["page_id"] for s in seen if not may_read(role, s["visibility"])]
        raw.append({"role": role, "filter": filter_role, "prompt": p, "retrieved": seen, "leaked": leaked,
                    "passed": not leaked})
    path = LEAK / f"raw-{label}.jsonl"
    path.write_text("".join(json.dumps(r) + "\n" for r in raw))
    return {"role": label, "prompts_run": len(raw), "passed": sum(r["passed"] for r in raw),
            "failed": sum(not r["passed"] for r in raw), "raw_output": f"[{path.name}]({path.name})"}

summary = pd.DataFrame([leak_run(r, r) for r in ROLES])
control = leak_run("student", "control-no-filter", filter_role=False)
summary.to_csv(LEAK / "summary.csv", index=False)
table = ["| Role | Prompts run | Passed | Failed | Raw output |", "| --- | --- | --- | --- | --- |"]
table += [f"| {r.role} | {r.prompts_run} | {r.passed} | {r.failed} | {r.raw_output} |" for r in summary.itertuples()]
(LEAK / "summary.md").write_text(
    f"# Leak test: Northwind roles\n\nRun on {date.today().isoformat()} with provider `{PROVIDER}`. Retrieval k = 5.\n\n"
    + "\n".join(table)
    + f"\n\nControl (filter off, as a student): {control['failed']} of {control['prompts_run']} prompts leaked, "
      f"as it should. Raw: {control['raw_output']}\n")
print((LEAK / "summary.md").read_text())
assert summary.failed.sum() == 0, "LEAK: a role retrieved a page it may not read"
assert control["failed"] > 0, "the test must be able to fail: with no filter, tutor or admin pages come back"
print("No leaks with the role filter on.")

# Leak test: Northwind roles

Run on 2026-11-15 with provider `mock`. Retrieval k = 5.

| Role | Prompts run | Passed | Failed | Raw output |
| --- | --- | --- | --- | --- |
| student | 16 | 16 | 0 | [raw-student.jsonl](raw-student.jsonl) |
| tutor | 16 | 16 | 0 | [raw-tutor.jsonl](raw-tutor.jsonl) |
| admin | 16 | 16 | 0 | [raw-admin.jsonl](raw-admin.jsonl) |

Control (filter off, as a student): 16 of 16 prompts leaked, as it should. Raw: [raw-control-no-filter.jsonl](raw-control-no-filter.jsonl)

No leaks with the role filter on.


Step 6: fill the field log

Every configuration choice, issue and workaround gets one row in `stage-11/field-log.csv`, tagged `configure`
(settings or content), `build` (a product change) or `drop` (not worth it), with the role or surface it hit.
The `item` column groups rows about the same thing, so a recurring item (2 or more rows) is easy to see.
Add your own rows by hand as you hit things: this is your log, not only the notebook's.

In [6]:
DECISIONS = {"configure", "build", "drop"}  # change settings or content / a product change / not worth it
COLUMNS = ["date", "item", "issue", "workaround", "decision", "role_or_surface", "note"]
today = date.today().isoformat()

# Questions the pages do not cover, one per role: the brief says these must end with "contact support".
OUT_OF_SCOPE = [("student", "Can I pay for lessons with a gift card from another shop?"),
                ("tutor", "Can I teach on my own video tool instead of the Northwind one?"),
                ("admin", "How do I export the company accounts to our tax software?")]
SUPPORT = re.compile(r"contact support", re.I)
gaps = [(role, q, answer(q, role, "out-of-scope")[0]) for role, q in OUT_OF_SCOPE]

entries = [{"date": today, "item": "access map written by hand", "issue": f"the {role} role has no setting: its page "
            "access is a Python dict in the notebook", "workaround": "wrote NW_ACCESS by hand", "decision": "configure",
            "role_or_surface": f"{role} / retrieval", "note": "every new customer repeats this step"} for role in ROLES]
for row in results[~results.correct].itertuples():
    entries.append({"date": today, "item": "question not answered from the right page",
                    "issue": f"{row.role} question not answered: {row.question}",
                    "workaround": "support answers it by email for now", "decision": "configure",
                    "role_or_surface": f"{row.role} / answers",
                    "note": f"expected {row.expected}, retrieved {row.retrieved}: check that page's visibility, "
                            "or write a page for this role"})
for role, q, text in gaps:
    if not SUPPORT.search(text):
        entries.append({"date": today, "item": "no contact support line", "issue": f"out of scope {role} question "
                        f"got no support pointer: {q}", "workaround": "none yet", "decision": "build",
                        "role_or_surface": f"{role} / answers", "note": f"answer was: {text[:80]}"})
entries += [
    {"date": today, "item": "no parent role", "issue": "Parents ask questions but there is no parent role",
     "workaround": "parents use the student login", "decision": "drop", "role_or_surface": "student / login",
     "note": "out of pilot scope; ask again after week 4"},
    {"date": today, "item": "sources are not links", "issue": "Admins want each answer to link to its help page",
     "workaround": "page ids shown as text", "decision": "build", "role_or_surface": "admin / chat widget",
     "note": "a product change: clickable sources in the widget, for every customer"},
]
log = pd.DataFrame(entries, columns=COLUMNS)
assert set(log.decision) <= DECISIONS, "decision must be configure, build or drop"
log.to_csv(OUTPUT / "field-log.csv", index=False)
recurring = log.item.value_counts()
recurring = recurring[recurring >= 2]
print("Wrote", len(log), "rows to", OUTPUT.name + "/field-log.csv")
print("Recurring items (2 or more rows):", recurring.to_dict())
assert len(recurring), "the brief needs at least one recurring item"
log

Wrote 10 rows to stage-11/field-log.csv
Recurring items (2 or more rows): {'access map written by hand': 3, 'no contact support line': 3, 'question not answered from the right page': 2}


,date,item,issue,workaround,decision,role_or_surface,note
0,2026-11-15,access map written by hand,the student role has no setting: its page acce...,wrote NW_ACCESS by hand,configure,student / retrieval,every new customer repeats this step
1,2026-11-15,access map written by hand,the tutor role has no setting: its page access...,wrote NW_ACCESS by hand,configure,tutor / retrieval,every new customer repeats this step
2,2026-11-15,access map written by hand,the admin role has no setting: its page access...,wrote NW_ACCESS by hand,configure,admin / retrieval,every new customer repeats this step
3,2026-11-15,question not answered from the right page,student question not answered: My tutor cancel...,support answers it by email for now,configure,student / answers,"expected ['nw-02'], retrieved ['nw-04', 'nw-03..."
4,2026-11-15,question not answered from the right page,student question not answered: Can my parent s...,support answers it by email for now,configure,student / answers,"expected ['nw-07'], retrieved ['nw-02', 'nw-04..."
5,2026-11-15,no contact support line,out of scope student question got no support p...,none yet,build,student / answers,answer was: The sources do not answer this.
6,2026-11-15,no contact support line,out of scope tutor question got no support poi...,none yet,build,tutor / answers,"answer was: To book a tutoring session, open t..."
7,2026-11-15,no contact support line,out of scope admin question got no support poi...,none yet,build,admin / answers,answer was: The sources do not answer this.
8,2026-11-15,no parent role,Parents ask questions but there is no parent role,parents use the student login,drop,student / login,out of pilot scope; ask again after week 4
9,2026-11-15,sources are not links,Admins want each answer to link to its help page,page ids shown as text,build,admin / chat widget,a product change: clickable sources in the wid...


Step 7: ship one change for a recurring item, with before and after

The example below takes the recurring item about the support line: when the pages do not answer, the answer must end
with a pointer to support. Put your chosen change in a file you commit (for example your answer function), and write
the commit hash and the before and after numbers in your README. The numbers are saved to `stage-11/change-evidence.csv`.

In [7]:
SUPPORT_LINE = "If this does not answer your question, contact support at support@northwind.example."

def answer_with_support(question, role, qid="adhoc"):
    text, cited, retrieved = answer(question, role, qid)
    if not cited and not SUPPORT.search(text):  # no page answered it: always give a way out
        text = f"{text} {SUPPORT_LINE}"
    return text, cited, retrieved

after = [(role, q, answer_with_support(q, role, "out-of-scope")[0]) for role, q in OUT_OF_SCOPE]
evidence = pd.DataFrame([{"role": role, "question": q, "before_has_support_line": bool(SUPPORT.search(b)),
                          "after_has_support_line": bool(SUPPORT.search(a)), "after_answer": a}
                         for (role, q, b), (_, _, a) in zip(gaps, after)])
evidence.to_csv(OUTPUT / "change-evidence.csv", index=False)
print(f"Out of scope answers with a support line: before {evidence.before_has_support_line.sum()} of {len(evidence)}, "
      f"after {evidence.after_has_support_line.sum()} of {len(evidence)}")
print("Files written:", sorted(str(p.relative_to(OUTPUT)) for p in OUTPUT.rglob("*") if p.is_file()))
evidence[["role", "before_has_support_line", "after_has_support_line"]]

Out of scope answers with a support line: before 0 of 3, after 2 of 3
Files written: ['answers.jsonl', 'audit-log.jsonl', 'change-evidence.csv', 'field-log.csv', 'leak-test/raw-admin.jsonl', 'leak-test/raw-control-no-filter.jsonl', 'leak-test/raw-student.jsonl', 'leak-test/raw-tutor.jsonl', 'leak-test/summary.csv', 'leak-test/summary.md']


,role,before_has_support_line,after_has_support_line
0,student,False,True
1,tutor,False,False
2,admin,False,True
